In [1]:
import os
import pandas as pd
import json
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.stats as stats

In [2]:
file_path = os.path.join("..","processed_data", "encoded_ai_company_adoption.csv")
df = pd.read_csv(file_path)
meta_path = os.path.join("..","processed_data", "meta_data.json")
df.head()

,survey_year,company_size,num_employees,annual_revenue_usd_millions,company_age,ai_adoption_rate,ai_adoption_stage,years_using_ai,num_ai_tools_used,ai_projects_active,...,ai_primary_tool_Other,ai_use_case_Fraud Detection,ai_use_case_HR Automation,ai_use_case_Marketing Automation,ai_use_case_Medical Diagnostics,ai_use_case_Predictive Maintenance,ai_use_case_Software Development,ai_use_case_Supply Chain Optimization,cost_reduction_percent,customer_satisfaction
0,2023,0,57,48.31,29,30.57,1,3,1,3,...,0,0,0,0,0,0,0,0,9.45,5.20
1,2023,0,57,48.31,29,27.25,1,4,3,0,...,0,0,0,0,0,0,1,0,0.00,6.98
2,2023,0,57,48.31,29,31.54,1,2,3,3,...,0,0,1,0,0,0,0,0,9.74,4.12
3,2023,0,57,48.31,29,11.02,1,2,1,2,...,0,0,0,0,0,0,1,0,0.00,5.72
4,2024,0,57,48.31,30,33.39,1,7,3,5,...,0,0,0,0,0,0,0,0,9.02,6.31


In [3]:
with open(meta_path, "r", encoding="utf-8") as f:
    meta_data = json.load(f)
    
    NUMERIC_ORDINAL_COLS = meta_data["numeric_ordinal_cols"]
    ONEHOT_COLS = meta_data["onehot_cols"]
    CYCLIC_COLS = meta_data["cyclic_cols"]
    BINARY_COLS = meta_data["binary_cols"]

if 'ai_failure_rate' in NUMERIC_ORDINAL_COLS:
    NUMERIC_ORDINAL_COLS.remove('ai_failure_rate')

In [4]:
y = df['ai_failure_rate']
X = df.drop(columns=['ai_failure_rate'])
X.shape, y.shape

((150000, 86), (150000,))

In [5]:
def preprocess_data(X_train, X_test):
    valid_num_cols = [col for col in NUMERIC_ORDINAL_COLS if col in X_train.columns]
    valid_cat_cols = [col for col in ONEHOT_COLS if col in X_train.columns]
    valid_cyc_cols = [col for col in CYCLIC_COLS if col in X_train.columns]

    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train[valid_num_cols].astype(float))
    X_test_num = scaler.transform(X_test[valid_num_cols].astype(float))

    X_train_cat = X_train[valid_cat_cols].astype(float).values
    X_test_cat = X_test[valid_cat_cols].astype(float).values

    train_col_means = X_train_cat.mean(axis=0)
    train_col_means = np.where(train_col_means == 0, 1e-8, train_col_means)

    X_train_cat_scaled = X_train_cat / np.sqrt(train_col_means)
    X_test_cat_scaled = X_test_cat / np.sqrt(train_col_means)

    X_train_cyc = X_train[valid_cyc_cols].astype(float).values
    X_test_cyc = X_test[valid_cyc_cols].astype(float).values

    X_train_combined = np.hstack([X_train_num, X_train_cat_scaled, X_train_cyc])
    X_test_combined = np.hstack([X_test_num, X_test_cat_scaled, X_test_cyc])

    final_columns = valid_num_cols + valid_cat_cols + valid_cyc_cols
    X_train_combined = pd.DataFrame(X_train_combined, columns=final_columns, index=X_train.index)
    X_test_combined = pd.DataFrame(X_test_combined, columns=final_columns, index=X_test.index)
    return X_train_combined, X_test_combined

In [8]:
def get_optimal_svd_components(X_train_scaled, target_variance=0.8, random_state=42, show_plot=True):
    max_cols = X_train_scaled.shape[1] - 1
    svd_test = TruncatedSVD(n_components=max_cols, random_state=random_state)
    svd_test.fit(X_train_scaled)
    
    #Tính phương sai tích lũy
    cumulative_variance = np.cumsum(svd_test.explained_variance_ratio_)
    
    #Tìm số chiều cắt tại ngưỡng target_variance
    optimal_components = np.argmax(cumulative_variance >= target_variance) + 1
    
    if show_plot:
        plt.figure(figsize=(8, 5))
        plt.plot(range(1, max_cols + 1), cumulative_variance, marker='o', linestyle='-', markersize=3, alpha=0.7)

        plt.axhline(y=target_variance, color='red', linestyle='--', linewidth=1.5, label=f'Ngưỡng {int(target_variance*100)}%')
        
        plt.axvline(x=optimal_components, color='green', linestyle='-.', linewidth=1.5, label=f'Tối ưu: {optimal_components}D')
        
        plt.xlabel('Số lượng components (Chiều)')
        plt.ylabel('Tỉ lệ thông tin tích lũy (Cumulative Variance)')
        plt.title(f'Phân tích thành phần chính (SVD) - Ngưỡng {int(target_variance*100)}%', fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
    return optimal_components

In [12]:
def run_rf_experiment_dynamic(X_data, y_data, method='Original'):
    splits = [0.2, 0.3, 0.4]
    results = []

    for test_size in splits:
        split_label = f"{int((1-test_size)*10)}:{int(test_size*10)}"

        # 1. Chia train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X_data, y_data, test_size=test_size, random_state=42
        )

        # 2. Chuẩn hóa
        X_train_combined, X_test_combined = preprocess_data(X_train, X_test)

        # 3. Giảm chiều
        if method.upper() == 'ORIGINAL':
            X_train_final, X_test_final = X_train_combined, X_test_combined
            data_type_name = 'Original'

        elif method.upper() == 'PCA':
            reducer = PCA(n_components=0.8, random_state=42)
            #reducer = PCA(n_components=0.8, svd_solver='full', random_state=42)
            X_train_final = reducer.fit_transform(X_train_combined)
            X_test_final  = reducer.transform(X_test_combined)
            data_type_name = 'PCA'

        elif method.upper() == 'SVD':
            dynamic_components = get_optimal_svd_components(
                X_train_combined, target_variance=0.8,
                random_state=42, show_plot=False
            )
            reducer = TruncatedSVD(n_components=dynamic_components, random_state=42)
            X_train_final = reducer.fit_transform(X_train_combined)
            X_test_final  = reducer.transform(X_test_combined)
            data_type_name = f'TruncatedSVD ({dynamic_components}D)'

        else:
            raise ValueError("method chỉ hỗ trợ 'Original', 'PCA' hoặc 'SVD'")


        # 4. Model
        y_train_arr = y_train.values if hasattr(y_train, 'values') else y_train

        model = RandomForestRegressor(n_estimators=100, max_depth=10,min_samples_split=10, min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1)
        model.fit(X_train_final, y_train_arr)
        
        y_train_pred = model.predict(X_train_final)
        y_test_pred  = model.predict(X_test_final)

        # 5. Đánh giá
        train_r2  = r2_score(y_train, y_train_pred)
        test_r2   = r2_score(y_test, y_test_pred)
        train_mae = mean_absolute_error(y_train, y_train_pred)
        test_mae  = mean_absolute_error(y_test, y_test_pred)
        train_mse = mean_squared_error(y_train, y_train_pred)
        test_mse  = mean_squared_error(y_test, y_test_pred)

        results.append({
            'Data Type':          data_type_name,
            'Split (Train:Test)': split_label,
            'Train R2':           train_r2,
            'Test R2':            test_r2,
            'Gap R2':             train_r2 - test_r2,
            'Train MAE':          train_mae,
            'Test MAE':           test_mae,
            'Train MSE':          train_mse,
            'Test MSE':           test_mse,

            'Model Object':       model,
            'X_train_raw':        X_train, 
            'X_test_raw':         X_test,  
            'y_test_actual':      y_test,
            'y_test_pred':        y_test_pred,
            'y_train_actual':     y_train,
            'y_train_pred':       y_train_pred
        })

    return results

In [13]:
res_orig = run_rf_experiment_dynamic(X, y, method='Original')
res_pca  = run_rf_experiment_dynamic(X, y, method='PCA')
res_svd  = run_rf_experiment_dynamic(X, y, method='SVD')

all_results = res_orig + res_pca + res_svd

metrics_keys = ['Data Type', 'Split (Train:Test)', 'Train R2', 'Test R2', 'Gap R2', 'Train MAE', 'Test MAE', 'Train MSE', 'Test MSE']
df_display = pd.DataFrame([{k: d[k] for k in metrics_keys} for d in all_results])

display(df_display.round(4))

,Data Type,Split (Train:Test),Train R2,Test R2,Gap R2,Train MAE,Test MAE,Train MSE,Test MSE
0,Original,8:2,0.6235,0.5939,0.0297,3.8382,3.9651,22.7052,24.3393
1,Original,7:3,0.6258,0.5970,0.0288,3.8255,3.9596,22.5754,24.1858
2,Original,6:4,0.6290,0.5961,0.0330,3.8082,3.9668,22.3582,24.3086
3,PCA,8:2,0.5053,0.4674,0.0378,4.4077,4.5580,29.8377,31.9161
4,PCA,7:3,0.5098,0.4711,0.0386,4.3875,4.5466,29.5738,31.7402
5,PCA,6:4,0.5095,0.4680,0.0415,4.3899,4.5628,29.5613,32.0126
6,TruncatedSVD (47D),8:2,0.4970,0.4586,0.0384,4.4436,4.5973,30.3362,32.4468
7,TruncatedSVD (47D),7:3,0.4971,0.4577,0.0394,4.4456,4.6037,30.3386,32.5493
8,TruncatedSVD (47D),6:4,0.5047,0.4623,0.0424,4.4123,4.5879,29.8546,32.3602
